# Online Shopper Purchase Intention — Portfolio Analysis
A clean, reproducible notebook for the portfolio version of the course project.

In [ ]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


## 1. Load the UCI dataset

In [ ]:
dataset = fetch_ucirepo(id=468)
X_raw = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].astype(int)
df = X_raw.copy()
df["Revenue"] = y
df = df.drop_duplicates().reset_index(drop=True)
df.shape, df["Revenue"].mean()


## 2. Feature engineering

In [ ]:
X = df.drop(columns="Revenue").copy()
y = df["Revenue"].astype(int)
categorical = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
numeric_cols = X.select_dtypes(include="number").columns
continuous = [c for c in numeric_cols if c not in categorical]
for col in continuous:
    X[col] = np.log1p(X[col])
X = pd.get_dummies(X, columns=categorical, drop_first=True, dtype=int)
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.80)]
X = X.drop(columns=to_drop)
X.shape


## 3. Train a Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = (model.predict_proba(X_test)[:, 1] > 0.62).astype(int)
print(classification_report(y_test, pred))
confusion_matrix(y_test, pred)


## 4. Feature importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importance.head(15)


## Next steps
See `ROADMAP.md` for calibration, SHAP explanations, additional models, and deployment ideas.